# Colab 24b — colab24 with a per-representation pool (no aa_len == ss_len)

**Identical to colab24 in every respect except the pool definition.** colab24 (and 22/23) kept only
domains where `aa_len == ss_len` — an AA<->SS *positional correspondence* requirement. Since every
edit distance here is computed **within** a representation (aa<->aa, ss<->ss, 3di<->3di) and we never
align AA position i to SS position i, that requirement is unnecessary and slightly arbitrary. colab24b
filters each representation **on its own terms** (standard alphabet + length in [MIN_LEN, MAX_LEN]).

This grows the pool ~4% (~10.1k -> ~10.5k) and will shift every downstream number slightly — it is a
deliberate, documented pool change, not a cosmetic one. colab24 is retained as the old (equal-length)
run of record for rollback.

Everything else is unchanged from colab24:
- Length-only retrieval baseline + bootstrap 95% CIs (Sections 9-13).
- Linear probes + PCA as primary space evidence, UMAP as illustration (Section 14).
- `encode_pad` raises on overflow; nominal-3-bin / near-degenerate-far-class framing; wording =
  exact-Levenshtein / string-similarity neighbours, never "biological".

## Note for colab25 (deferred — 2026-06-22)

**Curiosity probe for tomorrow:** artificially prop up the near-degenerate **far / low-sim bin** for
the **AA-trained** model. The far bin is empty here by construction (perturbation has an
alphabet-entropy floor ~0.28 ≈ BAND_LOW_AA 0.30), so the 3-bin head is effectively mid-vs-high.
colab25 = inject random-unrelated AA pairs (normLev ~0.10–0.28, the colab20 far-supplement generator)
into TRAINING to make a genuine 3-bin task, then re-run these colab24 retrieval + space controls to
see whether far-supervision changes AA retrieval or the embedding geometry. NOT motivated by an
observed failure (far pairs are already trivially separated geometrically) — a robustness/curiosity
probe. Decision today: comment-fix only in colab24; do not inject far pairs here.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_eval.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy umap-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Presentation palette (consistent across both charts)
COL_AA_ENC = '#3b7dd8'   # blue  = AA-encoder
COL_SS_ENC = '#36a85a'   # green = SS-encoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab19 / colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # high-sim cutoff (positives for AUROC; oracle true-set bar)
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10            # @k for both hits@10 and frac@10
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH} | retrieval @k={K_RETRIEVAL}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    # colab24 fix: raise instead of silently truncating. The pool is filtered to <= MAX_LEN,
    # so an overflow here means an unfiltered input slipped through — fail loudly, never alter
    # a protein string unnoticed.
    if len(seq) > max_len:
        raise ValueError(f'sequence length {len(seq)} exceeds max_len={max_len} '
                         f'(pool is filtered to <= {max_len}; unfiltered input?)')
    idx = [CHAR_TO_IDX[c] for c in seq]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    """Perturbation pairs: seed + k=round((1-t)*L) edits, t~U[0,1].
    NOTE (colab24 honesty fix): coverage is NOT uniform on [0,1]. Perturbing a single seed has an
    emergent alphabet-entropy FLOOR (realized edit distance saturates below the edit count as edits
    cancel under optimal alignment, plus chance alignment matches): normLev bottoms out ~0.28 for
    20-letter AA and ~0.49 for 3-letter SS. So pairs concentrate in mid/high; the 'far' (<BAND_LOW)
    bin is NEAR-DEGENERATE for AA (BAND_LOW_AA=0.30 sits right at the floor) and small for SS. This
    is a NOMINAL 3-bin setup whose practical learning signal is mid-vs-high. Populating 'far' would
    need a separate generator (random-unrelated pairs) -> deferred to colab25."""
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab19)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Data load — pool (colab24b: per-representation validity, NO aa_len == ss_len)

**Change vs colab24.** The dropped `aa_len == ss_len` line imposed an AA<->SS correspondence we never
use. Census on the committed data: it removed **615 / 14,907 (4.13%)** domains overall, **384** inside
the [50,200] AA range, and the mismatches are mostly tiny (median Δ = -1, IQR [-3, +1]) — benign
per-residue SS-annotation gaps, not corruption. colab24b instead filters each representation on its own
terms (standard alphabet + length in [MIN_LEN, MAX_LEN]). The independent SS/3Di length filters also
guard the new `encode_pad` overflow raise (7 mismatched in-range domains had ss_len > 200). The
length-only baseline already uses each representation's own sequence length, so it stays correct under
mismatched AA/SS lengths.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')

# colab24b: per-representation validity. We compute edit distance WITHIN a representation only, so
# AA<->SS positional correspondence is irrelevant -> NO aa_len==ss_len requirement. Each representation
# is filtered on its own terms: standard alphabet + length in [MIN_LEN, MAX_LEN].
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()
prot_df['ss_len'] = prot_df['ss_seq'].str.len()
RESCUED = {'4z0mC02', '3qkaE02'}
aa_in = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
ss_in = prot_df['ss_len'].between(MIN_LEN, MAX_LEN)   # independent SS length filter ->
                                                       # also guards encode_pad's new overflow raise
keep = (aa_in & ss_in) | prot_df['domain_id'].isin(RESCUED)
prot_df = prot_df[keep].reset_index(drop=True)
n_mismatch_kept = int((prot_df['aa_len'] != prot_df['ss_len']).sum())

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool (colab24b, per-rep validity): {len(prot_df)}  '
      f'({n_mismatch_kept} aa_len!=ss_len domains now retained vs colab24)')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))
# 3Di filtered on its own terms too: length guard so encode_pad never overflows on a 3Di feed.
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if len(s) <= MAX_LEN}
print(f'3Di sequences available (len <= {MAX_LEN}): {len(id_to_3di)}')

def pair_3di_score(a, b):
    if a in id_to_3di and b in id_to_3di:
        return norm_lev(id_to_3di[a], id_to_3di[b])
    return np.nan
eval_df['3di_score'] = [pair_3di_score(a, b) for a, b in zip(eval_df['domain_a'], eval_df['domain_b'])]

SCORE_COL = {'AA': 'aa_score', 'SS': 'ss_score', '3Di': '3di_score'}
LOOKUP    = {'AA': id_to_aa,  'SS': id_to_ss,  '3Di': id_to_3di}
for feed, sc in SCORE_COL.items():
    n = int((eval_df[sc] >= BAND_HIGH).sum())
    tag = 'powered' if n >= 200 else ('moderate' if n >= 30 else 'underpowered (n<30)')
    print(f'  {feed}-feed: {n} eval pairs >= {BAND_HIGH}  [{tag}]')

## 6. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 7. Metric machinery — predicted L2-sim, AUROC, eval-pair selection

- `pred_sim_strings`: deployment score `1 - ||e_a - e_b|| / 2` for arbitrary pairs.
- `encode_pool`: batch-encode the full retrieval pool once per (encoder, feed).
- `auroc_cell`: **Q1** number. is-high AUROC over the full eval set of a feed
  (positives = `>= 0.70`, negatives = the rest), discriminator = predicted L2-sim.
- `feed_high_pairs`: the `>= 0.70` eval pairs with both proteins in pool (the retrieval queries).

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def feed_eval_pairs(feed):
    """All eval pairs scored in this alphabet with both proteins available."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = eval_df.dropna(subset=[sc])
    return sub[sub['domain_a'].isin(lk) & sub['domain_b'].isin(lk)]

def feed_high_pairs(feed):
    """>= 0.70 eval pairs with both proteins in the retrieval pool (the queries)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    inpool = set(all_domains) & set(lk)
    sub = eval_df.dropna(subset=[sc])
    return sub[(sub[sc] >= BAND_HIGH) & sub['domain_a'].isin(inpool) & sub['domain_b'].isin(inpool)]

def auroc_cell(model, feed):
    """Q1 - is-high AUROC. Returns (auroc, n_pos, n_total)."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = feed_eval_pairs(feed)
    y = (sub[sc].values >= BAND_HIGH).astype(int)
    if y.sum() == 0 or y.sum() == len(y): return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

## 8. Retrieval — exact-Levenshtein oracle neighbour sets (uniform, no AA special-casing)

For each query we brute-force `normLev` of the query against all ~10k pool proteins and take
**everything `>= 0.70`** as the true neighbour set `T`. Same definition for AA, SS, 3Di — AA just
ends up with `|T|~1` because high-AA pairs are rare. `median |T|` is the crowding of each feed.

In [ ]:
def build_oracle_cache(feed):
    """Exact-Levenshtein true neighbour sets (>= BAND_HIGH) per query, over the full pool."""
    lk = LOOKUP[feed]
    pool_ids = [d for d in all_domains if d in lk]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]
    lens = np.array([len(s) for s in pool_seqs])
    sub = feed_high_pairs(feed)
    q_ids = sorted({d for pr in sub[['domain_a', 'domain_b']].values for d in pr})
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]
        denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed:<3}]: {len(q_ids):>3} queries, median |T|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets (all feeds, uniform)...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['AA', 'SS', '3Di']}

## 9. Retrieval metrics — per-query + bootstrap 95% CI, and a length-only ranker

Same oracle-set metric family as colab23, refactored to return **per-query** values so we can
(a) bootstrap 95% CIs over queries and (b) re-use the identical scoring for a **length-only**
baseline ranker (rank by `-|Δlength|`). The baseline answers: how much of the result is just
length-banding?

In [ ]:
K_LIST = (10, 50)
K_MAP  = 10
N_BOOT = 2000

def _ap_at_k(order, true_set, k):
    nt = len(true_set)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1; ap += hits / i
    return ap / min(nt, k)

def per_query_metrics(sim_fn, cache, k_list=K_LIST, k_map=K_MAP):
    """sim_fn(qi) -> similarity array over pool (higher = closer). Self is masked here.
    Returns per-query arrays for ap@k_map, R-prec, rec@k, prec@k, hit@k, nt."""
    idx = cache['idx']
    keys = ['ap', 'rprec', 'nt'] + [f'{m}@{k}' for k in k_list for m in ('rec', 'prec', 'hit')]
    per = {kk: [] for kk in keys}
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts)
        if nt == 0: continue
        sim = np.asarray(sim_fn(qi), dtype=float).copy(); sim[qi] = -np.inf
        order = np.argsort(-sim)
        per['nt'].append(nt)
        per['ap'].append(_ap_at_k(order, ts, k_map))
        per['rprec'].append(len(set(order[:nt].tolist()) & ts) / nt)
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts)
            per[f'rec@{k}'].append(ret / nt)
            per[f'prec@{k}'].append(ret / k)
            per[f'hit@{k}'].append(1.0 if ret >= 1 else 0.0)
    return {kk: np.asarray(v, float) for kk, v in per.items()}

def summarize_ci(per, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(per['nt'])
    out = {'n_q': n, 'median_n_true': float(np.median(per['nt'])) if n else np.nan}
    fields = {f'MAP@{K_MAP}': 'ap', 'Rprec': 'rprec', 'precision@10': 'prec@10',
              'recall@10': 'rec@10', 'recall@50': 'rec@50', 'hitrate@10': 'hit@10'}
    boot_idx = rng.integers(0, n, size=(n_boot, n)) if n >= 2 else None
    for name, kk in fields.items():
        arr = per[kk]; out[name] = float(arr.mean()) if n else np.nan
        if boot_idx is not None:
            b = arr[boot_idx].mean(axis=1)
            out[name + '_lo'] = float(np.percentile(b, 2.5))
            out[name + '_hi'] = float(np.percentile(b, 97.5))
        else:
            out[name + '_lo'] = out[name + '_hi'] = np.nan
    return out

def encoder_pool_np(model, feed):
    pe, _ = encode_pool(model, LOOKUP[feed], ORACLE[feed]['pool_ids'])
    return pe.cpu().numpy()

def length_pool(feed):
    return np.array([len(LOOKUP[feed][d]) for d in ORACLE[feed]['pool_ids']], dtype=float)

## 10. Compute — encoder retrieval (with CI) + length-only baseline

In [ ]:
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']
results = []

for enc_label, model in ENCODERS:
    for feed in FEEDS:
        auroc, n_pos, n_tot = auroc_cell(model, feed)
        emb = encoder_pool_np(model, feed)
        per = per_query_metrics(lambda qi, e=emb: 1.0 - np.linalg.norm(e - e[qi], axis=1) / 2.0, ORACLE[feed])
        s = summarize_ci(per)
        row = {'encoder': enc_label, 'feed': feed,
               'role': 'in-domain' if (enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')] else 'cross-rep',
               'auroc': auroc, 'auroc_n_pos': n_pos, 'auroc_n_total': n_tot}
        row.update(s); results.append(row)
        print(f"{enc_label} x {feed:<4} | AUROC={auroc:.3f} | "
              f"MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
              f"R-prec={s['Rprec']:.3f} [{s['Rprec_lo']:.3f},{s['Rprec_hi']:.3f}] | "
              f"prec@10={s['precision@10']:.3f} | med|T|={s['median_n_true']:.0f} (n_q={s['n_q']})")

print('\n--- length-only baseline (rank by -|delta length|) ---')
for feed in FEEDS:
    plen = length_pool(feed)
    per = per_query_metrics(lambda qi, p=plen: -np.abs(p - p[qi]), ORACLE[feed])
    s = summarize_ci(per)
    row = {'encoder': 'length-only', 'feed': feed, 'role': 'baseline',
           'auroc': np.nan, 'auroc_n_pos': 0, 'auroc_n_total': 0}
    row.update(s); results.append(row)
    print(f"length   x {feed:<4} | MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
          f"R-prec={s['Rprec']:.3f} | prec@10={s['precision@10']:.3f} | med|T|={s['median_n_true']:.0f}")

print('\n--- encoder MAP@10 vs length-only (how much is length-banding?) ---')
for feed in FEEDS:
    le = next(r['MAP@10'] for r in results if r['encoder'] == 'length-only' and r['feed'] == feed)
    for enc in ['AA-enc', 'SS-enc']:
        me = next(r['MAP@10'] for r in results if r['encoder'] == enc and r['feed'] == feed)
        verdict = 'encoder adds order signal' if (me - le) > 0.03 else '~length-explained'
        print(f'  {enc} x {feed:<4}: encoder {me:.3f} vs length {le:.3f} | lift {me - le:+.3f}  ({verdict})')

res_df = pd.DataFrame(results)

## 11. Chart 1 — Accuracy (AUROC is-high)

Unchanged from colab22. Grouped bars: blue = AA-encoder, green = SS-encoder, dashed = chance.
**AA bars are n_pos=5** (only 5 high-AA pairs exist) -> read AA as underpowered; SS (n=333) and
3Di (n=38) are the trustworthy columns.

In [ ]:
def grouped_vals(metric):
    return ([next(r[metric] for r in results if r['encoder']=='AA-enc' and r['feed']==f) for f in FEEDS],
            [next(r[metric] for r in results if r['encoder']=='SS-enc' and r['feed']==f) for f in FEEDS])

aa_auroc, ss_auroc = grouped_vals('auroc')
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_auroc, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_auroc, w, label='SS-encoder', color=COL_SS_ENC)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.text(len(FEEDS)-0.5, 0.515, 'chance (0.5)', color='grey', fontsize=9, ha='right')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('AUROC  (is-high >= 0.70)'); ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Accuracy: can embedding distance separate a high-similarity pair from a random one?')
for i, f in enumerate(FEEDS):
    npos = next(r['auroc_n_pos'] for r in results if r['feed']==f)
    ax.annotate(f'n_pos={npos}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='lower left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab23_accuracy_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Chart 2 — Retrieval quality (MAP@10) vs the length-only baseline, bootstrap 95% CI

Three bars per feed: AA-encoder (blue), SS-encoder (green), **length-only baseline (grey)**, with
bootstrap 95% CIs. Where an encoder bar clears the grey one, the embedding is contributing
sequence-order signal beyond length-banding; where they overlap, the result is length-explained.

In [ ]:
def val_ci(metric, enc):
    pts = np.array([next(r[metric] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    lo  = np.array([next(r[metric + '_lo'] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    hi  = np.array([next(r[metric + '_hi'] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    return pts, lo, hi

x = np.arange(len(FEEDS)); w = 0.27
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for off, enc, col, lab in [(-w, 'AA-enc', COL_AA_ENC, 'AA-encoder'),
                           (0.0, 'SS-enc', COL_SS_ENC, 'SS-encoder'),
                           (w, 'length-only', '#888888', 'length-only baseline')]:
    p, lo, hi = val_ci('MAP@10', enc)
    ax.bar(x + off, p, w, label=lab, color=col)
    ax.errorbar(x + off, p, yerr=np.vstack([p - lo, hi - p]), fmt='none', ecolor='black', capsize=3, lw=1)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel(f'MAP@{K_MAP}  (vs exact-Levenshtein neighbour set)')
ax.set_xlabel('Feed modality'); ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Retrieval quality vs a length-only baseline (bootstrap 95% CI)')
for i, f in enumerate(FEEDS):
    mt = next(r['median_n_true'] for r in results if r['feed'] == f)
    ax.annotate(f'med|T|={mt:.0f}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
plt.tight_layout(); plt.savefig('colab24_retrieval_vs_length.png', dpi=150, bbox_inches='tight'); plt.show()

## 13. Summary table + CSV — encoder retrieval (95% CI) + length baseline

In [ ]:
print('=' * 134)
print('COLAB24 SUMMARY — encoder retrieval (bootstrap 95% CI) + length-only baseline; oracle-set relevance')
print('=' * 134)
print(f'{"enc":<12}{"feed":<6}{"role":<11}{"AUROC":>7}{"MAP@10 [95% CI]":>24}{"R-prec [95% CI]":>24}'
      f'{"prec@10":>9}{"rec@10":>8}{"med|T|":>8}{"n_q":>6}')
print('-' * 134)
for r in results:
    a = f'{r["auroc"]:.3f}' if not np.isnan(r["auroc"]) else '-'
    mp = f'{r["MAP@10"]:.3f} [{r["MAP@10_lo"]:.3f},{r["MAP@10_hi"]:.3f}]'
    rp = f'{r["Rprec"]:.3f} [{r["Rprec_lo"]:.3f},{r["Rprec_hi"]:.3f}]'
    print(f'{r["encoder"]:<12}{r["feed"]:<6}{r["role"]:<11}{a:>7}{mp:>24}{rp:>24}'
          f'{r["precision@10"]:>9.3f}{r["recall@10"]:>8.3f}{r["median_n_true"]:>8.0f}{r["n_q"]:>6}')
res_df.to_csv('colab24_summary.csv', index=False)
print('\nSaved: colab24_summary.csv + colab24_{retrieval_vs_length,pca_spectrum,pca_truncation}.png')

## 14. Embedding-space analysis — linear probes + PCA (primary), UMAP (illustration)

Per the review, the **original-space linear evidence is primary**; UMAP is demoted to illustration
because its axis claims are nonlinear and orientation-arbitrary.

- **14.2 Linear probes** — held-out Ridge: is length (and composition) *linearly recoverable* from
  the 128-d embedding? Pairs with the Section-10 length-only baseline — the probe says length is
  *encoded*; the baseline says whether that encoding *explains retrieval*.
- **14.3 PCA spectrum + axis correlations** — intrinsic dimensionality (isotropy) + which linear axis
  carries length/composition.
- **14.4 PCA-truncation retrieval** — rate-distortion: MAP@10 vs # PCs kept.
- **14.5–14.8 UMAP, neighbour tables, displacement probe, zoom** — illustration; the neighbour
  tables are original-space and are the hard qualitative evidence.

All on the **AA encoder / AA feed** (the well-posed case).

In [ ]:
import umap
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# ---- composition helpers (alphabet-agnostic content descriptors) ----
def comp_hist(seq):
    v = np.zeros(len(AA_ALPHABET))
    for c in seq:
        j = CHAR_TO_IDX.get(c, -1)
        if j >= 0: v[j] += 1
    s = v.sum()
    return v / s if s > 0 else v

def shannon_entropy(seq):
    h = comp_hist(seq); h = h[h > 0]
    return float(-(h * np.log2(h)).sum()) if h.size else 0.0

# ---- substrate: AA-encoder pool embedding + 2D UMAP + descriptors ----
VIZ_MODEL = model_aa
pool_ids_viz = [d for d in all_domains if d in id_to_aa]
idx_viz = {d: i for i, d in enumerate(pool_ids_viz)}
pool_emb_viz, _ = encode_pool(VIZ_MODEL, id_to_aa, pool_ids_viz)
pool_emb_viz = pool_emb_viz.cpu().numpy()
print(f'pool embeddings: {pool_emb_viz.shape}')

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
pool_2d = reducer.fit_transform(pool_emb_viz)
id_to_xy = {d: pool_2d[i] for i, d in enumerate(pool_ids_viz)}

lens_viz = np.array([len(id_to_aa[d]) for d in pool_ids_viz])
comp_mat = np.stack([comp_hist(id_to_aa[d]) for d in pool_ids_viz])          # (N, 20)
ent_viz  = np.array([shannon_entropy(id_to_aa[d]) for d in pool_ids_viz])
comp_pc  = PCA(n_components=2, random_state=SEED).fit_transform(comp_mat)     # composition axes

HIGH_PAIRS_AA = [tuple(p) for p in feed_high_pairs('AA')[['domain_a', 'domain_b']].values]
print(f'high-AA pairs for overlay: {len(HIGH_PAIRS_AA)}')

### 14.2 Linear probes — what is *linearly recoverable* from the 128-d embedding?

Held-out 5-fold Ridge. A high length-R² makes "length is linearly recoverable" a sharp claim; read
it together with the Section-10 length-only retrieval baseline (encoded ≠ load-bearing).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error

def linear_probe(emb, target, n_splits=5, seed=SEED):
    kf = KFold(n_splits, shuffle=True, random_state=seed)
    pred = np.zeros(len(target), dtype=float)
    for tr, te in kf.split(emb):
        pred[te] = Ridge(alpha=1.0).fit(emb[tr], target[tr]).predict(emb[te])
    return float(r2_score(target, pred)), float(mean_absolute_error(target, pred))

r2_len, mae_len = linear_probe(pool_emb_viz, lens_viz.astype(float))
r2_pc1, mae_pc1 = linear_probe(pool_emb_viz, comp_pc[:, 0])

kf = KFold(5, shuffle=True, random_state=SEED)
pred_h = np.zeros_like(comp_mat)
for tr, te in kf.split(pool_emb_viz):
    pred_h[te] = Ridge(alpha=1.0).fit(pool_emb_viz[tr], comp_mat[tr]).predict(pool_emb_viz[te])
r2_hist = float(r2_score(comp_mat, pred_h, multioutput='variance_weighted'))

print('Held-out linear probe (Ridge, 5-fold) — linear recoverability from the 128-d AA-encoder embedding')
print(f'  length                       : R2 = {r2_len:6.3f}   MAE = {mae_len:5.1f} residues')
print(f'  composition PC1              : R2 = {r2_pc1:6.3f}   MAE = {mae_pc1:5.3f}')
print(f'  full AA-composition histogram: R2 = {r2_hist:6.3f}   (variance-weighted, 20-d)')
print('\n-> length is (near-)linearly recoverable; composition only partially.')
print('   Section 10 length-only baseline tests whether that recoverability EXPLAINS retrieval')
print('   or is encoded-but-not-load-bearing.')

### 14.3 PCA — variance spectrum (intrinsic dimensionality) + linear axis correlations

Is the representation low-rank or distributed/isotropic? And which *linear* axis carries length vs
composition (the harder version of the colab23 UMAP-axis table).

In [ ]:
pca = PCA(n_components=min(128, pool_emb_viz.shape[1]), random_state=SEED).fit(pool_emb_viz)
evr = pca.explained_variance_ratio_; cum = np.cumsum(evr)
d90 = int(np.searchsorted(cum, 0.90) + 1); d99 = int(np.searchsorted(cum, 0.99) + 1)
pcs = pca.transform(pool_emb_viz)

print('PCA variance spectrum (AA-encoder pool embedding)')
print(f'  PC1 {evr[0]:.3f} | PC2 {evr[1]:.3f} | PC1+2 {cum[1]:.3f}')
print(f'  components for 90% var: {d90} ; for 99%: {d99}  (of {len(evr)})')
print(f'  -> {"low-rank" if d90 <= 8 else "distributed / isotropic"} representation')

def _rho(a, b): return float(spearmanr(a, b).correlation)
print('\nLinear (PCA) axis correlations:')
print(f'{"":<14}{"PC1":>8}{"PC2":>8}{"PC3":>8}')
for name, vec in [('length', lens_viz), ('comp-PC1', comp_pc[:, 0]), ('entropy', ent_viz)]:
    print(f'{name:<14}{_rho(pcs[:, 0], vec):>8.3f}{_rho(pcs[:, 1], vec):>8.3f}{_rho(pcs[:, 2], vec):>8.3f}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(np.arange(1, len(cum) + 1), cum, marker='o', ms=3)
ax.axhline(0.9, ls='--', color='grey', lw=1); ax.axvline(d90, ls=':', color='grey', lw=1)
ax.set_xscale('log', base=2)
ax.set_xlabel('# principal components'); ax.set_ylabel('cumulative explained variance')
ax.set_title('PCA variance spectrum (AA-encoder pool embedding)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('colab24_pca_spectrum.png', dpi=150, bbox_inches='tight'); plt.show()

### 14.4 PCA-truncation retrieval — rate-distortion (MAP@10 vs embedding rank)

Project pool embeddings onto the top-`d` PCs and recompute MAP@10. Saturation at small `d` = the
edit-distance retrieval signal is low-rank; a slow climb = distributed. AA feed (n_q=10) is
illustrative; SS feed is powered.

In [ ]:
D_GRID = [1, 2, 4, 8, 16, 32, 64, 128]

def trunc_map(model, feed, d_grid=D_GRID):
    emb = encoder_pool_np(model, feed)
    dmax = min(max(d_grid), emb.shape[1])
    full = PCA(n_components=dmax, random_state=SEED).fit_transform(emb)
    out = []
    for d in d_grid:
        sub = full[:, :d]
        per = per_query_metrics(lambda qi, s=sub: -np.linalg.norm(s - s[qi], axis=1), ORACLE[feed])
        out.append(float(per['ap'].mean()) if len(per['ap']) else np.nan)
    return out

map_aa = trunc_map(model_aa, 'AA')
map_ss = trunc_map(model_aa, 'SS')

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(D_GRID, map_aa, marker='o', color=COL_AA_ENC, label='AA feed (n_q=10, illustrative)')
ax.plot(D_GRID, map_ss, marker='s', color='#9467bd', label='SS feed (powered)')
ax.set_xscale('log', base=2); ax.set_xlabel('# PCA components kept'); ax.set_ylabel(f'MAP@{K_MAP}')
ax.set_title('Rate-distortion: retrieval quality vs embedding rank (AA-encoder)')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('colab24_pca_truncation.png', dpi=150, bbox_inches='tight'); plt.show()
print('truncation MAP@10  AA:', [f'{v:.2f}' for v in map_aa])
print('truncation MAP@10  SS:', [f'{v:.2f}' for v in map_ss])

### 14.5 UMAP figure — illustration only (high-AA pairs as red segments)

Kept for intuition; the axis claims here are the *soft* evidence — the linear probes (14.2-14.3) are
the hard version.

In [ ]:
ux, uy = pool_2d[:, 0], pool_2d[:, 1]
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(ux, uy, c=lens_viz, cmap='viridis', s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Protein length (AA)')
for a, b in HIGH_PAIRS_AA:
    if a in id_to_xy and b in id_to_xy:
        xa, ya = id_to_xy[a]; xb, yb = id_to_xy[b]
        ax.plot([xa, xb], [ya, yb], color='red', lw=1.5, alpha=0.9, zorder=5)
        ax.scatter([xa, xb], [ya, yb], color='red', s=80, edgecolors='black', linewidths=1.2, zorder=6)
        ax.annotate(f'{a[:6]}\u2194{b[:6]}', ((xa+xb)/2, (ya+yb)/2), fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='red', alpha=0.8))
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title('AA-encoder pool embedding (UMAP) — high-AA partner pairs as red segments')
plt.tight_layout(); plt.savefig('colab23_umap_pool.png', dpi=150, bbox_inches='tight'); plt.show()

### 14.6 Neighbourhood tables — original-space, the hard qualitative evidence

For each high-AA query, the top embedding neighbours with exact normLev + composition-sim. Shows the
true partner ranks #1 amid length-twins that are NOT exact-Levenshtein-similar.

In [ ]:
N_NB = 12
def neighbours_table(query, partner):
    qi = idx_viz[query]
    esim = 1.0 - np.linalg.norm(pool_emb_viz - pool_emb_viz[qi], axis=1) / 2.0
    esim[qi] = -np.inf
    order = np.argsort(-esim)[:N_NB]
    qpart_nl = norm_lev(id_to_aa[query], id_to_aa[partner])
    print(f'\nQuery {query} (len {len(id_to_aa[query])}) | partner {partner} '
          f'(exact normLev {qpart_nl:.3f})')
    print(f'  {"rank":<5}{"domain":<10}{"emb-sim":>8}{"len":>5}{"normLev":>9}{"comp-sim":>9}  note')
    for rk, oi in enumerate(order, 1):
        d = pool_ids_viz[oi]
        nl = norm_lev(id_to_aa[query], id_to_aa[d])
        cs = float((comp_mat[qi] * comp_mat[oi]).sum() /
                   max(1e-9, np.linalg.norm(comp_mat[qi]) * np.linalg.norm(comp_mat[oi])))
        note = '<== TRUE PARTNER' if d == partner else ('high-sim' if nl >= BAND_HIGH else '')
        print(f'  {rk:<5}{d:<10}{esim[oi]:>8.3f}{len(id_to_aa[d]):>5}{nl:>9.3f}{cs:>9.3f}  {note}')

for a, b in HIGH_PAIRS_AA[:3]:
    neighbours_table(a, b)

### 14.7 Horizontal-vs-vertical displacement probe — UMAP illustration

What changes when you move along each UMAP axis (length vs same-length content). Illustration; read
alongside the linear axis correlations in 14.3.

In [ ]:
rng_viz = np.random.default_rng(SEED)
N_PAIRS = 30000
ii = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
jj = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
ok = ii != jj; ii, jj = ii[ok], jj[ok]

dx   = np.abs(ux[ii] - ux[jj])
dy   = np.abs(uy[ii] - uy[jj])
dlen = np.abs(lens_viz[ii] - lens_viz[jj]).astype(float)

# composition distance, vectorised: 1 - cosine on the precomputed histograms
ca, cb = comp_mat[ii], comp_mat[jj]
na = np.linalg.norm(ca, axis=1); nb = np.linalg.norm(cb, axis=1)
denom = np.clip(na * nb, 1e-9, None)
dcomp = 1.0 - (ca * cb).sum(1) / denom

ratio = (dx + 1e-9) / (dy + 1e-9)
horiz = ratio > np.quantile(ratio, 0.8)   # top 20% most-horizontal displacements
vert  = ratio < np.quantile(ratio, 0.2)   # top 20% most-vertical

print('Displacement-direction probe (what changes when you move along each axis)')
print(f'{"group":<22}{"mean d-length":>14}{"mean d-comp":>13}')
print(f'{"mostly-horizontal":<22}{dlen[horiz].mean():>14.1f}{dcomp[horiz].mean():>13.3f}')
print(f'{"mostly-vertical":<22}{dlen[vert].mean():>14.1f}{dcomp[vert].mean():>13.3f}')
print(f'{"all pairs":<22}{dlen.mean():>14.1f}{dcomp.mean():>13.3f}')

same_len = dlen <= 3
print(f'\nAmong near-same-length pairs (|d-length| <= 3, n={int(same_len.sum())}):')
print(f'  corr(|dx|, d-comp) = {np.corrcoef(dx[same_len], dcomp[same_len])[0, 1]:+.3f}')
print(f'  corr(|dy|, d-comp) = {np.corrcoef(dy[same_len], dcomp[same_len])[0, 1]:+.3f}')
print('  -> at fixed length, the axis whose displacement tracks d-comp is the content axis;')
print('     i.e. two same-length sequences with totally different characters separate along THAT axis.')

### 14.8 Local zoom — illustration

One pair's neighbourhood coloured by exact normLev-to-query: the crowd of length-twins made visible.

In [ ]:
qa, qb = HIGH_PAIRS_AA[0]
cx, cy = id_to_xy[qa]
span_x = (ux.max() - ux.min()) * 0.12
span_y = (uy.max() - uy.min()) * 0.12
near = (np.abs(ux - cx) < span_x) & (np.abs(uy - cy) < span_y)
near_idx = np.where(near)[0]
nl_to_q = np.array([norm_lev(id_to_aa[qa], id_to_aa[pool_ids_viz[i]]) for i in near_idx])

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(ux[near_idx], uy[near_idx], c=nl_to_q, cmap='magma', vmin=0, vmax=1,
                s=30, alpha=0.85, linewidths=0)
plt.colorbar(sc, ax=ax, label=f'exact normLev to query {qa}')
xa, ya = id_to_xy[qa]; xb, yb = id_to_xy[qb]
ax.plot([xa, xb], [ya, yb], color='red', lw=2, zorder=5)
ax.scatter([xa, xb], [ya, yb], color='red', s=130, edgecolors='black', linewidths=1.3, zorder=6)
ax.annotate('query', (xa, ya), fontsize=9, color='red')
ax.annotate('partner', (xb, yb), fontsize=9, color='red')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title(f'Local neighbourhood of {qa} <-> {qb}\n(most nearby points are length-twins with low normLev)')
plt.tight_layout(); plt.savefig('colab23_umap_zoom.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'local window: {len(near_idx)} points; '
      f'{int((nl_to_q >= BAND_HIGH).sum())} of them are genuine high-sim (>= {BAND_HIGH})')

## How to read this notebook

**Controls (Sections 9-13).** The retrieval numbers now carry **bootstrap 95% CIs** and sit next to
a **length-only baseline**. The decisive read is per feed: if the encoder MAP@10 bar clears the grey
length-only bar (non-overlapping CIs), the embedding contributes **sequence-order** signal beyond
length-banding; if they overlap, that feed's retrieval is **length-explained**. Expectation: AA
clears it decisively (length-twins are not exact-Levenshtein neighbours); SS/3Di is the open question
the baseline settles.

**Space (Section 14).** Linear probes + PCA are primary: the held-out length R² states how much length
is *linearly encoded*, and the length-only baseline states whether that encoding *drives retrieval*.
PCA gives intrinsic dimensionality and a rate-distortion curve. UMAP and the displacement probe are
illustration; the original-space neighbour tables are the hard qualitative evidence.

**Wording.** Relevance is the **exact-Levenshtein / high string-similarity neighbour set**; no
biological claim is made. The head is a **nominal 3-bin classifier with a near-degenerate far class**
(practical signal = mid-vs-high); see the make_full_range_pairs note and the colab25 deferral.